# NB10 - Slice-Based Development
## Background
A model's average test accuracy conceals failures on important subgroups. Slice-based evaluation (Chen et al., 2019 — Slice Finder; Chung et al., 2019 — SliceLine) identifies cohesive data subsets where the model performs poorly. These slices often correspond to demographics, edge cases, or distribution shifts that matter disproportionately for deployment.

The clinical ML community calls this 'subgroup analysis' and it is now required by regulatory bodies (FDA, EMA) for AI-based medical devices. Responsible AI frameworks at major tech companies mandate slice analysis before deployment. Snorkel's AI platform built a business around programmatic slice identification.

## What You'll Learn
- Defining slices via predicates on features and metadata
- Finding failure slices: automated exhaustive search over predicate space
- Slice-level metrics: accuracy, coverage, severity score
- Targeted data collection strategy based on failure slices

## Why This Matters
Aggregate accuracy is a necessary but not sufficient metric. A model with 95% overall accuracy might perform at 60% on a critical demographic. Slice analysis prevents these failures from reaching production.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Callable
from dataclasses import dataclass

np.random.seed(42)

# ── Synthetic dataset with structured failure modes ───────────────────────
n_samples = 1200
n_classes = 3

# Metadata slices
age_group     = np.random.choice(['young', 'middle', 'senior'], n_samples, p=[0.4, 0.4, 0.2])
data_source   = np.random.choice(['device_A', 'device_B', 'device_C'], n_samples, p=[0.5, 0.3, 0.2])
time_of_day   = np.random.choice(['morning', 'afternoon', 'night'], n_samples, p=[0.4, 0.4, 0.2])

# Features
X = np.random.randn(n_samples, 6)

# True labels with class imbalance
y_true = np.random.choice(n_classes, n_samples, p=[0.5, 0.3, 0.2])

# Model predictions: good overall but fails on specific slices
y_pred = y_true.copy()
# Failure mode 1: senior + night -> frequent errors
senior_night = (age_group == 'senior') & (time_of_day == 'night')
y_pred[senior_night] = np.random.choice(n_classes, senior_night.sum())

# Failure mode 2: device_C -> higher error rate
device_c = data_source == 'device_C'
flip_mask = device_c & (np.random.rand(n_samples) < 0.4)
y_pred[flip_mask] = (y_true[flip_mask] + 1) % n_classes

# Failure mode 3: Rare class on device_A morning
rare_slice = (y_true == 2) & (data_source == 'device_A') & (time_of_day == 'morning')
y_pred[rare_slice] = np.random.choice(n_classes, rare_slice.sum())

overall_acc = (y_pred == y_true).mean()
print(f'Overall accuracy: {overall_acc:.3f}')
print('(Slices with injected failures: senior+night, device_C, rare_class+deviceA+morning)')

In [ ]:
# ── Slice Finder: exhaustive predicate search ─────────────────────────────
@dataclass
class Slice:
    predicates: Dict[str, str]
    accuracy: float
    coverage: int
    severity: float  # deviation from overall accuracy

def evaluate_slice(mask: np.ndarray, y_pred: np.ndarray, y_true: np.ndarray
                   ) -> Tuple[float, int]:
    n = mask.sum()
    if n < 10:  # Minimum slice size
        return 1.0, n
    acc = (y_pred[mask] == y_true[mask]).mean()
    return float(acc), int(n)

def find_failure_slices(metadata: Dict[str, np.ndarray],
                         y_pred: np.ndarray, y_true: np.ndarray,
                         overall_acc: float,
                         min_coverage: int = 10,
                         max_severity: float = 0.1) -> List[Slice]:
    """Enumerate all single and pairwise predicates to find failure slices."""
    failure_slices = []

    keys = list(metadata.keys())
    all_values = {k: list(set(metadata[k])) for k in keys}

    # Single predicates
    for key in keys:
        for val in all_values[key]:
            mask = metadata[key] == val
            acc, n = evaluate_slice(mask, y_pred, y_true)
            severity = overall_acc - acc
            if n >= min_coverage and severity > max_severity:
                failure_slices.append(Slice({key: val}, acc, n, severity))

    # Pairwise predicates
    for i, k1 in enumerate(keys):
        for k2 in keys[i+1:]:
            for v1 in all_values[k1]:
                for v2 in all_values[k2]:
                    mask = (metadata[k1] == v1) & (metadata[k2] == v2)
                    acc, n = evaluate_slice(mask, y_pred, y_true)
                    severity = overall_acc - acc
                    if n >= min_coverage and severity > max_severity:
                        failure_slices.append(
                            Slice({k1: v1, k2: v2}, acc, n, severity))

    failure_slices.sort(key=lambda s: s.severity, reverse=True)
    return failure_slices

metadata = {
    'age_group':   age_group,
    'data_source': data_source,
    'time_of_day': time_of_day,
}

slices = find_failure_slices(metadata, y_pred, y_true, overall_acc, max_severity=0.05)

print(f'Found {len(slices)} failure slices (severity > 5pp below overall):')
print()
print(f'{'Predicates':40s} | {'Coverage':>9} | {'Accuracy':>9} | {'Severity':>9}')
print('-' * 75)
for s in slices[:10]:
    pred_str = ', '.join(f'{k}={v}' for k,v in s.predicates.items())
    print(f'{pred_str:40s} | {s.coverage:>9d} | {s.accuracy:>9.3f} | {s.severity:>9.3f}')

In [ ]:
# ── Visualize failure slices ───────────────────────────────────────────────
if slices:
    top_n = min(8, len(slices))
    top_slices = slices[:top_n]
    labels = [', '.join(f'{k}={v}' for k,v in s.predicates.items()) for s in top_slices]
    severities = [s.severity for s in top_slices]
    coverages = [s.coverage for s in top_slices]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    colors = ['red' if sv > 0.2 else 'orange' for sv in severities]
    axes[0].barh(range(top_n), severities, color=colors, alpha=0.8)
    axes[0].set_yticks(range(top_n))
    axes[0].set_yticklabels(labels, fontsize=8)
    axes[0].axvline(0.1, color='red', linestyle='--', label='High severity threshold')
    axes[0].set_title('Failure Slices by Severity')
    axes[0].set_xlabel('Accuracy drop vs overall'); axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='x')

    axes[1].scatter(coverages, severities, c=colors, s=80, alpha=0.8)
    for i, label in enumerate(labels):
        axes[1].annotate(label, (coverages[i], severities[i]),
                         fontsize=7, ha='left', va='bottom')
    axes[1].axhline(0.1, color='red', linestyle='--', alpha=0.5, label='High severity')
    axes[1].set_title('Slice Coverage vs Severity')
    axes[1].set_xlabel('Slice size (samples)'); axes[1].set_ylabel('Severity (acc drop)')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{BASE}/s29_10_slices.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Priority: collect more data for high-severity slices (top-left region)')